In [ ]:
# Module 10 — Code Along: the LLM as a task engine (support-ticket triage)
# SHARED DAY-4 BLOCK — domain + injected fake LLM. Paste once, runs everywhere (M10-M12).
from __future__ import annotations
import json
from types import SimpleNamespace
from typing import Optional, Any, Callable
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, ValidationError

@dataclass
class SupportTicket:
    id: int
    subject: str
    category: str          # "Billing" | "Technical" | "Account"
    priority: str          # "low" | "normal" | "high" | "urgent"
    is_open: bool = True
    tags: list[str] = field(default_factory=list)

SAMPLE_TICKETS = [
    SupportTicket(1, "Can't log in after password reset", "Account", "high"),
    SupportTicket(2, "Invoice double-charged this month",  "Billing", "urgent"),
    SupportTicket(3, "How do I export my data?",            "Technical", "low"),
    SupportTicket(4, "App crashes on upload",               "Technical", "high"),
    SupportTicket(5, "Refund not received",                 "Billing", "normal", is_open=False),
]

# --- the dependency-injection seam: a fake stand-in for openai.OpenAI ---
def _msg(content=None, tool_calls=None):
    return SimpleNamespace(content=content, tool_calls=tool_calls)
def _resp(message):
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])
def _tool_call(call_id, name, **args):
    return SimpleNamespace(id=call_id,
        function=SimpleNamespace(name=name, arguments=json.dumps(args)))

class FakeLLM:
    """Scripts chat.completions.create - same shape as openai.OpenAI, no API key.
    Exactly the Day-3 pattern: inject a fake at the seam instead of the real client."""
    def __init__(self, responses):
        self._responses = list(responses)
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        return self._responses.pop(0)

print(f"{len(SAMPLE_TICKETS)} tickets loaded; FakeLLM ready (no API key needed)")

# Section 1 · Closed-source API power

One hosted engine, many tasks. Same `chat.completions.create` call — the **task lives in the prompt**. We run on `FakeLLM` here (no key needed); the shape is identical to a real OpenAI client.

Tasks below: **classify** a ticket's priority, **extract** fields, **summarize** a thread.

In [ ]:
# TASK = CLASSIFY -- subject in, structured label out.
# Script the FakeLLM to answer with JSON (a real model returns text; we parse it).
llm = FakeLLM([
    _resp(_msg(content='{"priority": "urgent"}')),
])

subject = "Invoice double-charged this month"
prompt = f"Classify this support ticket's priority (low|normal|high|urgent). "\
         f"Reply as JSON {{\"priority\": ...}}. Subject: {subject!r}"

resp = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
)
raw = resp.choices[0].message.content      # the model speaks text...
label = json.loads(raw)                     # ...we parse it into structure
print("raw   ->", raw)
print("label ->", label["priority"])        # urgent

## TASK = summarize — collapse a thread to its gist

A support thread can be dozens of turns. Summarize compresses it to one actionable line an agent can scan. Same pattern: thread in, short text out — here we keep the output as plain text (no schema needed).

In [ ]:
# TASK = SUMMARIZE -- a multi-turn thread in, one line out.
thread = [
    "Customer: I was charged twice for my June invoice.",
    "Agent: I see two charges of $49 on the 3rd. Let me check.",
    "Customer: Yes, please refund the duplicate.",
    "Agent: Refund issued, 3-5 business days.",
]

llm = FakeLLM([
    _resp(_msg(content="Duplicate $49 June charge refunded; ETA 3-5 business days.")),
])

resp = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user",
               "content": "Summarise this support thread in one line:\n" + "\n".join(thread)}],
)
print("SUMMARY:", resp.choices[0].message.content)

### Same shape, different vendor (Gemini)

This whole section would look identical against Google's Gemini API — a model id, a list of messages, a response object. We stay on OpenAI for M11/M12, but the *pattern* is portable. (No live Gemini call here — cameo only.)

# Section 2 · Open-source LLM power

Models you run yourself — no API key, data stays local. `transformers.pipeline()` wraps load→preprocess→infer→postprocess into one call.

Two cells run **live** on small CPU models (set `RUN_LIVE_HF=1` to actually download+run). Three more are **shown, not run** — the models are multi-GB, so we display real outputs without making the room wait.

In [ ]:
import os
RUN_LIVE_HF = os.getenv("RUN_LIVE_HF") == "1"   # gate downloads; off by default

# --- LIVE: zero-shot classification (no training, you supply the labels) ---
if RUN_LIVE_HF:
    from transformers import pipeline
    clf = pipeline("zero-shot-classification",
                   model="typeform/distilbert-base-uncased-mnli")
    out = clf("My invoice is wrong and I was double charged",
              candidate_labels=["billing", "hardware", "account", "urgent"])
    print(list(zip(out["labels"], [round(s, 3) for s in out["scores"]])))
else:
    print("RUN_LIVE_HF!=1 — skipping live download. Expected output:")
    print("[('billing', 0.94), ('account', 0.04), ('urgent', 0.01), ('hardware', 0.01)]")

In [ ]:
# --- LIVE: named-entity recognition (pull people/orgs/places out of text) ---
if RUN_LIVE_HF:
    from transformers import pipeline
    ner = pipeline("ner", model="dslim/bert-base-NER", grouped_entities=True)
    ents = ner("Tim Cook from Apple called about the Cupertino order.")
    print([(e["word"], e["entity_group"]) for e in ents])
else:
    print("RUN_LIVE_HF!=1 — skipping live download. Expected output:")
    print("[('Tim Cook', 'PER'), ('Apple', 'ORG'), ('Cupertino', 'LOC')]")

### Shown, not run — heavier modalities

These are the same one-line `pipeline()` calls; the models are multi-GB so we show real outputs instead of downloading live.

```python
# Speech → text (meeting transcribe)
pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")("meeting.flac")
# >>> {'text': 'GOING ALONG SLUSHY COUNTRY ROADS AND SPEAKING TO DAMP AUDIENCES ...'}

# Image → caption
pipeline("image-to-text", model="Salesforce/blip-image-captioning-base")("parrots.png")
# >>> [{'generated_text': 'two birds are standing next to each other'}]

# Translation EN→ES
pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")("Hello, how are you today?")
# >>> [{'translation_text': 'Hola, ¿cómo estás hoy?'}]
```

# Section 3 · Your first LLM app

Now the part Lab 10 makes you build: take a raw call and make it **structured + validated**. The LLM is just another untrusted source — validate at the boundary, exactly like a Day-2 request body.

## Structured output — make the LLM speak your schema

A free-text answer is hard to use; a **schema** turns the LLM into a typed function. Two levers:

- **Pydantic model** = the contract. You describe each field (and its `description=`) and the model becomes your parse target.
- **JSON mode** — pass `response_format={"type": "json_object"}` to the real client so the API guarantees valid JSON. (Our `FakeLLM` ignores it; mentioned so you recognise it in real code.)

The schema does double duty: it's the *instruction* to the model **and** the *validator* on the way back.

In [ ]:
# The extraction target: a Pydantic model. Every field's description doubles as
# the instruction we can hand the model (and JSON-mode honours the schema).
class TicketQuery(BaseModel):
    category: Optional[str] = Field(default=None, description="Restrict to this category, or null for all.")
    priority: Optional[str] = Field(default=None, description="One of low|normal|high|urgent, or null.")
    open_only: bool = False
    subject_contains: Optional[str] = Field(default=None, description="Substring the subject must contain.")

# model_json_schema() is exactly what you'd send as the function/JSON-mode schema.
print(json.dumps(TicketQuery.model_json_schema(), indent=2))

## Always validate — the LLM is just another untrusted source

Same boundary discipline as Day 2's HTTP bodies and CSV rows: **data crossing into your system is guilty until validated.** The LLM is no different — it can return the wrong shape, an extra field, or plain broken JSON.

So we never trust `message.content`. We feed it straight through `TicketQuery.model_validate_json(...)` and let Pydantic accept it or raise. Garbage in -> `ValidationError`, caught -> handled. Never a half-parsed dict leaking downstream.

In [ ]:
# TASK = EXTRACT + VALIDATE  (<- deck: module-10 §3.3)
# Natural language in, a validated TicketQuery out -- the same NL->structured task
# you build by hand in Lab 10's parse_nl_query.
llm = FakeLLM([
    # 1) a well-formed extraction
    _resp(_msg(content='{"category": "Billing", "open_only": true}')),
    # 2) a malformed reply (truncated JSON) -- the LLM misbehaving
    _resp(_msg(content='{"category": "Billing", "open_only":')),
])

def extract_query(user_text: str) -> TicketQuery:
    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},   # real client honours this; FakeLLM ignores it
        messages=[{"role": "user", "content": f"Extract a TicketQuery from: {user_text!r}"}],
    )
    raw = resp.choices[0].message.content
    return TicketQuery.model_validate_json(raw)     # validate at the boundary

# Good case: parses into a typed object
q = extract_query("show me open billing tickets")
print("OK ->", q)

# Bad case: malformed JSON is caught, not crashed
try:
    extract_query("anything")
except ValidationError as e:
    print("ValidationError caught:", e.error_count(), "error(s) -- we never trusted it")

## The improvement ladder (concept — shown, not run)

Output not good enough? Climb only as far as the task needs:

| Rung | Fix it by… | When |
|---|---|---|
| **Prompt** | clearer role / constraints / examples | first, always |
| **RAG** | retrieve facts, inject as context | model lacks *your* knowledge |
| **Fine-tune** | retrain weights on labeled data | one narrow task, high volume, fixed format |

Fine-tuning is one call — `client.fine_tuning.jobs.create(training_file=..., model="gpt-4o-mini")` — fed a JSONL of `(prompt → expected JSON)` pairs, then **kept only if it beats the base model on a held-out set**. That held-out check is exactly the **golden-eval you build in M12**. We don't run it today.

## Recap — Module 10

- **§1 Closed API** — one hosted engine, many tasks (classify / extract / summarize); same shape across vendors
- **§2 Open-source** — `pipeline()` runs a task locally, no key; pick closed for reasoning, open for a fixed high-volume task
- **§3 Your first app** — structured output + **validate at the boundary**; climb the prompt→RAG→fine-tune ladder only as needed

→ **Lab 10**: Part A classify a Product (open-source), Part B build `parse_nl_query` + `apply_query`.
→ **M11**: wrap this one call in a loop with tools.